# 05 - Pipelines de Fraude en Producción: Desbalanceo Extremo, Curvas PR-AUC y Arquitecturas Híbridas

## Curso de Machine Learning: Modelos No Supervisados

En sistemas reales de procesamiento de pagos (Visa, Mastercard, SPEI, Stripe), el fraude representa apenas entre el **$0.05\%$ y el $0.5\%$ del volumen transaccional**. En escenarios de desbalanceo tan extremo:
1. **La Exactitud (*Accuracy*) y el ROC-AUC son engañosos**: Un modelo que genere un $2\%$ de falsos positivos tendrá un ROC-AUC de $0.98$, pero inundará la bandeja de los analistas con $90\%$ de falsas alarmas.
2. **La métrica reina es el Área Bajo la Curva Precision-Recall (PR-AUC / Average Precision)**.
3. **Arquitectura Híbrida (No Supervisado + Supervisado)**: Una de las técnicas ganadoras en competencias y producción bancaria consiste en utilizar el puntaje continuo de `IsolationForest` y la distancia de reconstrucción de `PCA` como **nuevas variables predictoras (Feature Engineering)** para alimentar un modelo supervisado (Regresión Logística / Gradient Boosting).

---

### Contenidos
1. Por qué el ROC-AUC Engaña en Desbalanceo Extremo y por qué Usar **PR-AUC**
2. Entrenamiento Semi-Supervisado de Novedades (*Novelty Detection*): Entrenar Solo con Transacciones Limpias
3. Arquitectura Híbrida en Producción: `IsolationForest` Score como Feature para un Clasificador Supervisado
4. Optimización del Umbral por Matriz de Costos Financieros (Costo de Fraude vs. Costo de Fricción/Revisión)
5. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    precision_recall_curve, confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Simulación de 15,000 Transacciones con Desbalanceo Extremo (Solo 0.8% de Fraude = 120 casos)
rng = np.random.default_rng(42)
n_norm = 14880
n_fr = 120

X_n = rng.normal(0, 1.0, size=(n_norm, 8))
# Fraude sutil (mezcla de fraude extremo y fraude camuflado)
X_f1 = rng.normal(2.6, 0.9, size=(80, 8))
X_f2 = rng.uniform(-4.5, 4.5, size=(40, 8))

X_all = np.vstack([X_n, X_f1, X_f2])
y_all = np.array([0]*n_norm + [1]*n_fr)

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.30, random_state=42, stratify=y_all)
print(f"Train: {len(X_tr)} tx ({y_tr.sum()} fraudes, {y_tr.mean()*100:.2f}%)")
print(f"Test:  {len(X_te)} tx ({y_te.sum()} fraudes, {y_te.mean()*100:.2f}%)")


## 1. Arquitectura Híbrida: Isolation Forest como Generador de Features

Comparamos tres enfoques sobre el conjunto de prueba (`X_te`):
1. **Isolation Forest Puro (100% No Supervisado)**.
2. ** Clasificador Supervisado Estándar (`RandomForestClassifier`)** entrenado sobre las 8 variables originales.
3. **Modelo Híbrido (Supervisado + Feature No Supervisada)**: Añadimos como novena columna el puntaje `-iforest.score_samples(X)` para dotar al clasificador supervisado de visión global de aislamiento.


In [ ]:
# 1. Isolation Forest No Supervisado
if_prod = IsolationForest(n_estimators=200, max_samples=256, random_state=42, n_jobs=-1)
if_prod.fit(X_tr)

score_if_tr = -if_prod.score_samples(X_tr).reshape(-1, 1)
score_if_te = -if_prod.score_samples(X_te).reshape(-1, 1)

# 2. Modelo Supervisado solo con variables originales
rf_base = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_base.fit(X_tr, y_tr)
prob_base = rf_base.predict_proba(X_te)[:, 1]

# 3. Modelo Híbrido (Variables Originales + Score de Isolation Forest)
X_tr_hib = np.hstack([X_tr, score_if_tr])
X_te_hib = np.hstack([X_te, score_if_te])

rf_hibrido = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf_hibrido.fit(X_tr_hib, y_tr)
prob_hibrido = rf_hibrido.predict_proba(X_te_hib)[:, 1]

# Comparación de Curvas Precision-Recall
plt.figure(figsize=(10, 5.5))
for nombre, sc, col in [
    ('1. Isolation Forest Puro (No Supervisado)', score_if_te.ravel(), '#ff7f0e'),
    ('2. Random Forest Base (Solo Vars Originales)', prob_base, '#1f77b4'),
    ('3. Arquitectura Híbrida (Vars + Isolation Score)', prob_hibrido, '#2ca02c')
]:
    prec, rec, _ = precision_recall_curve(y_te, sc)
    pr_auc = average_precision_score(y_te, sc)
    roc = roc_auc_score(y_te, sc)
    plt.plot(rec, prec, lw=2.3, color=col, label=f'{nombre} | PR-AUC = {pr_auc:.3f} (ROC = {roc:.3f})')

plt.xlabel('Recall (Fracción de Fraudes Totales Capturados)', fontsize=11, fontweight='bold')
plt.ylabel('Precision (Pureza de las Alertas Emitidas)', fontsize=11, fontweight='bold')
plt.title('Curvas Precision-Recall bajo Desbalanceo Extremo (0.8% Fraude)', fontsize=12.5, fontweight='bold')
plt.legend(fontsize=10, loc='lower left')
plt.tight_layout()
plt.show()


## 2. Optimización Financiera por Matriz de Costos en Dólares

En la vida real, cada error tiene un costo financiero distinto:
- **Falso Negativo (Fraude no detectado)**: El banco pierde el monto íntegro de la transacción fraudulenta (ej. promedio de **$850 USD** por fraude).
- **Falso Positivo (Falsa alarma / Bloqueo a cliente legítimo)**: Costo operativo de llamada del Call Center e insatisfacción del cliente (ej. **$15 USD** por revisión).

Busquemos el umbral del puntaje de Isolation Forest que **minimiza la pérdida total en dólares**.


In [ ]:
costo_fraude_no_detectado = 850.0  # USD por Falso Negativo
costo_revision_alerta = 15.0       # USD por cada Alerta enviada a revisión (TP + FP)

umbrales = np.percentile(score_if_te, np.linspace(90, 99.8, 100))
costos_totales = []

for u in umbrales:
    alerta = (score_if_te.ravel() >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, alerta).ravel()
    costo_u = (fn * costo_fraude_no_detectado) + ((tp + fp) * costo_revision_alerta)
    costos_totales.append(costo_u)

idx_min = np.argmin(costos_totales)
umbral_opt = umbrales[idx_min]
costo_sin_modelo = y_te.sum() * costo_fraude_no_detectado
ahorro_neto = costo_sin_modelo - costos_totales[idx_min]

plt.figure(figsize=(10, 4.8))
plt.plot(umbrales, costos_totales, color='crimson', lw=2.5, label='Costo Financiero Total (FN + Revisiones)')
plt.axvline(umbral_opt, color='black', linestyle='--', label=f'Umbral Óptimo = {umbral_opt:.3f}')
plt.axhline(costo_sin_modelo, color='gray', linestyle=':', label=f'Pérdida Sin Modelo (${costo_sin_modelo:,.0f} USD)')
plt.xlabel('Umbral de Corte sobre Score de Isolation Forest', fontsize=11, fontweight='bold')
plt.ylabel('Costo Total para el Banco (USD)', fontsize=11, fontweight='bold')
plt.title(f'Optimización Financiera del Umbral: Ahorro Neto = ${ahorro_neto:,.0f} USD', fontsize=12, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Conclusiones

1. **PR-AUC sobre ROC-AUC**: En detección de fraude (<1% de casos positivos), evalúa siempre tus modelos mediante `average_precision_score` (PR-AUC).
2. **Arquitectura Híbrida**: Inyectar el puntaje continuo de `IsolationForest` como una columna adicional en un clasificador supervisado combina la detección de patrones conocidos con la sensibilidad a anomalías nunca antes vistas.
3. **Umbral Basado en Costos (ROI)**: Fija el punto de corte minimizando la función de pérdida económica real del negocio.
